In [ ]:
%load_ext autoreload
%autoreload 2

%load_ext rich

In [ ]:
import json
import os
import random
from collections import defaultdict
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown
from rich import print

from aymurai.utils.json_data import load_json

sns.set_theme(style="whitegrid")
plt.rcParams.update(
    {"figure.figsize": (10, 6), "axes.titlesize": 14, "axes.labelsize": 12}
)

## Load annotations

In [ ]:
annotations_path = "/resources/annotations/label-studio/resos-annotations/30-nov/project-3-at-2022-11-30-16-04-2b43bf39.json"

In [ ]:
annotations = load_json(annotations_path)
print(f"Loaded {len(annotations)} annotations")

In [ ]:
def collect_label_annotations(annotations, target_labels):
    """Build minimal evaluation buckets keyed by label."""
    buckets_by_label = defaultdict(list)
    target_labels = set(target_labels)

    for task in annotations:
        for annotation in task.get("annotations", []):
            merged = {}
            for result in annotation.get("result", []):
                result_id = result.get("id")
                value = result.get("value", {})
                slot = merged.setdefault(
                    result_id,
                    {
                        "text": value.get("text"),
                        "start": value.get("start"),
                        "end": value.get("end"),
                        "labels": [],
                        "choices": [],
                    },
                )

                # Update span info if present in the current result.
                slot["text"] = slot["text"] or value.get("text")
                slot["start"] = (
                    slot["start"] if slot["start"] is not None else value.get("start")
                )
                slot["end"] = (
                    slot["end"] if slot["end"] is not None else value.get("end")
                )
                slot["labels"].extend(value.get("labels", []))
                slot["choices"].extend(value.get("choices", []))

            for payload in merged.values():
                labels = [
                    label for label in payload["labels"] if label in target_labels
                ]
                if not labels or payload["text"] is None:
                    continue

                item = {
                    "text": payload["text"],
                    "labels": list(dict.fromkeys(payload["labels"])),
                    "choices": list(dict.fromkeys(payload["choices"])),
                    "start": payload["start"],
                    "end": payload["end"],
                }

                for label in labels:
                    buckets_by_label[label].append(deepcopy(item))

    return {label: buckets_by_label.get(label, []) for label in target_labels}

In [ ]:
target_labels = [
    "CONDUCTA",
    "CONDUCTA_DESCRIPCION",
    "DETALLE",
    "OBJETO_DE_LA_RESOLUCION",
]
samples_by_label = collect_label_annotations(annotations, target_labels)

## Exploratory data analysis

We consolidate the annotated spans into a tabular view to inspect label and subcategory coverage before running any retrieval experiments.


In [ ]:
def samples_to_dataframe(samples):
    rows = []
    for label, items in samples.items():
        for item in items:
            rows.append(
                {
                    "label": label,
                    "text": item["text"],
                    "labels": item["labels"],
                    "choices": item["choices"],
                    "n_choices": len(item["choices"]),
                    "char_len": len(item["text"]),
                }
            )
    return pd.DataFrame(rows)


samples_df = samples_to_dataframe(samples_by_label)
samples_df.head()

In [ ]:
samples_df["labels"].map(len).describe()

In [ ]:
samples_df["choices"].map(len).describe()

In [ ]:
samples_df.loc[samples_df["n_choices"] == 0]

In [ ]:
# Remove samples with zero choices
samples_df = samples_df[samples_df["n_choices"] > 0].reset_index(drop=True)
samples_df

In [ ]:
for label, samples in samples_by_label.items():
    print(f"{label}: {len(samples)} samples before filtering")
    samples_by_label[label] = [
        sample for sample in samples if len(sample["choices"]) > 0
    ]
    print(f"{label}: {len(samples_by_label[label])} samples after filtering")

In [ ]:
samples_df.loc[samples_df["n_choices"] > 1, ["text", "choices"]].explode(
    "choices"
).drop_duplicates()

In [ ]:
label_counts = samples_df.groupby("label").size().sort_values(ascending=False)
choice_counts = (
    samples_df.explode("choices").groupby(["label", "choices"]).size().rename("count")
)

summary_table = pd.DataFrame(
    {
        "samples": label_counts,
        "unique_choices": choice_counts.groupby("label").size(),
        "avg_text_len": samples_df.groupby("label")["char_len"].mean().round(1),
    }
)
summary_table

In [ ]:
fig, ax = plt.subplots()
order = label_counts.index
sns.barplot(x=label_counts.values, y=label_counts.index, ax=ax, palette="viridis")
ax.set_title("Samples per label")
ax.set_xlabel("Number of samples")
ax.set_ylabel("Label")
plt.tight_layout()
plt.show()

plot_data = (
    choice_counts.reset_index()
    .sort_values("count", ascending=False)
    .groupby("label")
    .head(10)
)

for label in order:
    subset = plot_data.query("label == @label").sort_values("count", ascending=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=subset, x="count", y="choices", palette="viridis", ax=ax)
    ax.set_title(f"Top subcategories for {label}")
    ax.set_xlabel("Frequency")
    ax.set_ylabel("Subcategory")
    ax.grid(axis="x", alpha=0.2)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=samples_df, x="label", y="char_len", palette="viridis")
ax.set_title("Entity span length by label")
ax.set_xlabel("Label")
ax.set_ylabel("Character length")
ax.set_ylim(0, 100)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
for label in label_counts.index:
    print(f"=== {label} ===")
    subset = samples_df.query("label == @label")
    for _, row in subset.sample(n=min(3, len(subset))).iterrows():
        print(f"- Text: {row['text']}")
        print(f"  Choices: {row['choices']}")
    print()

## USEM pipeline audit

We validate the existing TensorFlow-based USEM subcategorizer by reloading the cached assets, recomputing embeddings, and measuring retrieval accuracy before swapping in PyTorch alternatives.

In [ ]:
pipeline_path = Path("/resources/pipelines/production/datapublic/pipeline.json")

with pipeline_path.open() as f:
    pipeline_config = json.load(f)

usem_configs = {
    config[1]["category"]: {
        "subcategories_path": config[1]["subcategories_path"],
        "response_embeddings_path": config[1]["response_embeddings_path"],
    }
    for config in pipeline_config["postprocess"]
    if "USEMSubcategorizer" in config[0]
}

usem_configs

In [ ]:
from aymurai.transforms.entity_subcategories.usem import USEMSubcategorizer

usem_models = {}
for label, cfg in usem_configs.items():
    print(f"Loading USEM assets for {label}…")
    usem_models[label] = USEMSubcategorizer(category=label, **cfg)

list(usem_models.keys())

In [ ]:
model = usem_models["CONDUCTA"]

In [ ]:
model

In [ ]:
model.subcategories

In [ ]:
# Sample encoding of first subcategory
encoded = model.usem.encode(
    [model.subcategories[0].replace("_", " ")],
    context_array=[model.subcategories[0].replace("_", " ")],
    encoder_type="response_encoder",
)
encoded

In [ ]:
def cosine_similarity(a, b):
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    return np.sum(a_norm * b_norm, axis=1)

In [ ]:
# Compute cosine similarity matrix - first value should be 1
cosine_similarity(encoded, model.usem_vectors)

In [ ]:
import unicodedata


def normalize_text(text: str) -> str:
    """Normalize text by lowercasing and removing non-alphanumeric characters."""
    text = text.lower()
    text = "".join(
        char
        for char in unicodedata.normalize("NFKD", text)
        if char.isalnum() or char.isspace()
    )
    return text


def normalize_subcategory(name: str) -> str:
    """Mirror the preprocessing applied when the cached embeddings were built."""
    # name = normalize_text(name)
    return name.replace("_", " ")


def recompute_response_vectors(model):
    """Return normalized subcategory strings and freshly encoded vectors."""
    normalized = [
        normalize_subcategory(subcategory) for subcategory in model.subcategories
    ]
    fresh_vectors = model.usem.encode(
        normalized,
        context_array=normalized,
        encoder_type="response_encoder",
    )
    return normalized, fresh_vectors.numpy()


cached = model.usem_vectors
normalized_subcategories, fresh = recompute_response_vectors(model)

In [ ]:
def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    """Row-wise L2 normalization to prepare cosine similarity checks."""
    return matrix / np.linalg.norm(matrix, axis=1, keepdims=True)


cached_norm = l2_normalize(cached)
fresh_norm = l2_normalize(fresh)
cosine_matrix = cached_norm @ fresh_norm.T
cosine_matrix.shape

In [ ]:
row_max_indices = np.argmax(cosine_matrix, axis=1)
row_max_values = cosine_matrix[np.arange(len(row_max_indices)), row_max_indices]
mismatched_rows = np.where(row_max_indices != np.arange(len(row_max_indices)))[0]
diag_values = np.diag(cosine_matrix)
off_identity_matches = (
    pd.DataFrame(
        {
            "subcategory": [model.subcategories[idx] for idx in mismatched_rows],
            "matched_index": row_max_indices[mismatched_rows],
            "max_cosine": row_max_values[mismatched_rows],
            "diagonal_cosine": diag_values[mismatched_rows],
        }
    )
    if mismatched_rows.size
    else pd.DataFrame(
        columns=[
            "subcategory",
            "matched_index",
            "max_cosine",
            "diagonal_cosine",
        ]
    )
)
off_identity_matches  # .head()

In [ ]:
diag_values

In [ ]:
tolerance = 0.99
near_identity_mask = diag_values >= tolerance
print(
    f"Diagonal entries at or above {tolerance:.2f}: {near_identity_mask.sum()} / {len(diag_values)}"
)
diagonal_gaps = (
    pd.DataFrame(
        {
            "subcategory": [
                model.subcategories[idx]
                for idx, ok in enumerate(near_identity_mask)
                if not ok
            ],
            "diagonal_cosine": diag_values[~near_identity_mask],
            "shortfall": 1 - diag_values[~near_identity_mask],
        }
    )
    if (~near_identity_mask).any()
    else pd.DataFrame(columns=["subcategory", "diagonal_cosine", "shortfall"])
)
diagonal_gaps  # .head()

In [ ]:
embedding_checks = []
for label, model in usem_models.items():
    cached = model.usem_vectors
    normalized_subcategories, fresh = recompute_response_vectors(model)
    cached_norm = l2_normalize(cached)
    fresh_norm = l2_normalize(fresh)
    delta = cached - fresh
    row_l2 = np.linalg.norm(delta, axis=1)
    cos = np.sum(cached_norm * fresh_norm, axis=1)
    embedding_checks.append(
        {
            "label": label,
            "max_l2": float(row_l2.max()),
            "mean_l2": float(row_l2.mean()),
            "min_cos": float(cos.min()),
            "mean_cos": float(cos.mean()),
        }
    )

embedding_checks_df = pd.DataFrame(embedding_checks).set_index("label")
embedding_checks_df

In [ ]:
# Using batch retrieval
def retrieve_samples(samples, models, top_k=5):
    records = []
    for label, items in samples.items():
        model = models[label]
        texts = [item["text"] for item in items]
        retrieved_lists = model.batch_retrieve(texts, top_k=top_k)
        for item, retrieved in zip(items, retrieved_lists):
            records.append(
                {
                    "label": label,
                    "text": item["text"],
                    "choices": item["choices"],
                    "retrieved": retrieved,
                }
            )
    return records


retrieval_records = retrieve_samples(samples_by_label, usem_models, top_k=5)
len(retrieval_records)

In [ ]:
# Filter out samples with no annotated choices
retrieval_records = [record for record in retrieval_records if record["choices"]]
len(retrieval_records)

In [ ]:
def compute_topk_accuracy(records, ks=(1, 2, 3, 4, 5)):
    metrics = []
    labels = sorted({record["label"] for record in records})
    for label in labels:
        label_records = [record for record in records if record["label"] == label]
        total = len(label_records)
        for k in ks:
            hits = sum(
                any(choice in record["retrieved"][:k] for choice in record["choices"])
                for record in label_records
            )
            metrics.append(
                {
                    "label": label,
                    "k": k,
                    "accuracy": hits / total if total else np.nan,
                }
            )

    overall = []
    for k in ks:
        hits = sum(
            any(choice in record["retrieved"][:k] for choice in record["choices"])
            for record in records
        )
        overall.append({"label": "OVERALL", "k": k, "accuracy": hits / len(records)})

    metrics.extend(overall)
    return pd.DataFrame(metrics)


baseline_accuracy = compute_topk_accuracy(retrieval_records)
baseline_accuracy_pivot = baseline_accuracy.pivot(
    index="label", columns="k", values="accuracy"
).sort_index()

baseline_accuracy_pivot

In [ ]:
fig, ax = plt.subplots()

sns.heatmap(
    baseline_accuracy_pivot.loc[
        [label for label in baseline_accuracy_pivot.index if label != "OVERALL"]
    ],
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    ax=ax,
)
ax.set_title("USEM baseline top-k accuracy by label")
ax.set_xlabel("k")
ax.set_ylabel("Label")
plt.tight_layout()
plt.show()

baseline_accuracy_pivot.loc[["OVERALL"]]

In [ ]:
# For each label, show some random success cases where the correct choice was retrieved in top 5
for label in target_labels:
    print(f"=== {label} ===")
    label_records = [
        record
        for record in retrieval_records
        if record["label"] == label
        and any(choice in record["retrieved"][:5] for choice in record["choices"])
    ]
    for record in random.sample(label_records, k=min(3, len(label_records))):
        print(f"- Text: {record['text']}")
        print(f"  Choices: {record['choices']}")
        print(f"  Retrieved: {record['retrieved'][:5]}")
    print()

In [ ]:
# For each label, show some random failure cases where the correct choice was not retrieved in top 5
for label in target_labels:
    print(f"=== {label} ===")
    label_records = [
        record
        for record in retrieval_records
        if record["label"] == label
        and not any(choice in record["retrieved"][:5] for choice in record["choices"])
    ]
    for record in random.sample(label_records, k=min(3, len(label_records))):
        print(f"- Text: {record['text']}")
        print(f"  Choices: {record['choices']}")
        print(f"  Retrieved: {record['retrieved'][:5]}")
    print()

In [ ]:
overall = baseline_accuracy_pivot.loc["OVERALL"]
label_top1 = baseline_accuracy_pivot.loc[
    [label for label in baseline_accuracy_pivot.index if label != "OVERALL"], 1
]

summary_md = f"""
### Baseline findings

- Overall top-1 accuracy: {overall[1]:.2%}
- Overall top-3 accuracy: {overall[3]:.2%}
- Overall top-5 accuracy: {overall[5]:.2%}
- Best-performing label (top-1): {label_top1.idxmax()} at {label_top1.max():.2%}
- Most challenging label (top-1): {label_top1.idxmin()} at {label_top1.min():.2%}

#### Next steps
1. Replace the TensorFlow encoder with sentence-transformer checkpoints and repeat the evaluation.
2. Compare latency and memory footprint during retrieval for each contender.
3. Consolidate the winning model into the pipeline and update deployment assets.
"""

Markdown(summary_md)

## Random Retriever

In [ ]:
class RandomRetriever:
    def __init__(self, subcategories: list[str], seed: int = 42):
        self.subcategories = subcategories
        self._rng = random.Random(seed)

    def batch_retrieve(self, texts: list[str], top_k: int = 5) -> list[list[str]]:
        k = min(top_k, len(self.subcategories))
        if k == 0:
            return [[] for _ in texts]
        return [self._rng.sample(self.subcategories, k) for _ in texts]

In [ ]:
label_subcategories = {
    label: model.subcategories for label, model in usem_models.items()
}
random_models = {
    label: RandomRetriever(subcats, seed=idx * 17)
    for idx, (label, subcats) in enumerate(label_subcategories.items())
}

In [ ]:
baseline_variants = {
    "Random": random_models,
    # "USEM": usem_models,
    # "BM25": bm25_models,
    # "USEM + BM25 hybrid": hybrid_models,
}

variant_records = {}
for variant, models in baseline_variants.items():
    records = retrieve_samples(samples_by_label, models, top_k=5)
    variant_records[variant] = records

variant_accuracy = {
    variant: compute_topk_accuracy(records)
    for variant, records in variant_records.items()
}
variant_pivots = {
    variant: df.pivot(index="label", columns="k", values="accuracy").sort_index()
    for variant, df in variant_accuracy.items()
}

for variant, pivot in variant_pivots.items():
    display(Markdown(f"**{variant} baseline top-k accuracy**"))
    display(pivot)

# combined_series = {
#     "TensorFlow USEM": baseline_accuracy_pivot.loc["OVERALL"],
#     "DistilUSE": distiluse_accuracy_pivot.loc["OVERALL"],
#     "MiniLM": minilm_accuracy_pivot.loc["OVERALL"],
# }
# combined_series.update(
#     {variant: pivot.loc["OVERALL"] for variant, pivot in variant_pivots.items()}
# )
# combined_accuracy = pd.DataFrame.from_dict(combined_series, orient="index")
# combined_accuracy

In [ ]:
fig, ax = plt.subplots()

sns.heatmap(
    variant_pivots["Random"].loc[
        [label for label in baseline_accuracy_pivot.index if label != "OVERALL"]
    ],
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    ax=ax,
)
ax.set_title("Random baseline top-k accuracy by label")
ax.set_xlabel("k")
ax.set_ylabel("Label")
plt.tight_layout()
plt.show()

variant_pivots["Random"].loc[["OVERALL"]]

## BM25 retriever

In [ ]:
import math
import re
from collections import Counter

token_pattern = re.compile(r"\w+", re.UNICODE)


def tokenize(text: str) -> list[str]:
    return token_pattern.findall(normalize_text(text))


class BM25Retriever:
    def __init__(self, subcategories: list[str], k1: float = 1.2, b: float = 0.75):
        self.subcategories = subcategories
        normalized = [normalize_subcategory(sub) for sub in subcategories]
        self.tokenized = [tokenize(text) for text in normalized]
        self.doc_len = [len(tokens) for tokens in self.tokenized]
        self.avgdl = sum(self.doc_len) / len(self.doc_len)
        self.k1 = k1
        self.b = b
        self.N = len(self.tokenized)
        self.doc_freqs = [Counter(tokens) for tokens in self.tokenized]
        corpus_tokens = set().union(*self.doc_freqs)
        self.df = Counter(
            {
                token: sum(1 for doc in self.doc_freqs if token in doc)
                for token in corpus_tokens
            }
        )
        self.idf = {
            token: math.log(1 + (self.N - freq + 0.5) / (freq + 0.5))
            for token, freq in self.df.items()
        }

    def _score(self, tokens: list[str]) -> np.ndarray:
        scores = np.zeros(self.N, dtype=np.float64)
        for token in tokens:
            idf = self.idf.get(token)
            if idf is None:
                continue
            for idx, freq_map in enumerate(self.doc_freqs):
                freq = freq_map.get(token, 0)
                if freq == 0:
                    continue
                denom = freq + self.k1 * (
                    1 - self.b + self.b * self.doc_len[idx] / self.avgdl
                )
                scores[idx] += idf * freq * (self.k1 + 1) / denom
        return scores

    def batch_retrieve(self, texts: list[str], top_k: int = 5) -> list[list[str]]:
        return [self._topk(text, top_k) for text in texts]

    def _topk(self, text: str, top_k: int) -> list[str]:
        scores = self._score(tokenize(text))
        if top_k <= 0:
            return []
        top_indices = np.argsort(-scores)[:top_k]
        return [self.subcategories[idx] for idx in top_indices]

    def score_vector(self, text: str) -> np.ndarray:
        return self._score(tokenize(text))


class HybridRetriever:
    def __init__(self, usem_model, bm25_model, bm25_weight: float = 0.5):
        self.usem_model = usem_model
        self.bm25_model = bm25_model
        self.response_vectors = usem_model.usem_vectors
        self.bm25_weight = bm25_weight

    def batch_retrieve(self, texts: list[str], top_k: int = 5) -> list[list[str]]:
        query_vectors = self.usem_model.usem.batch_encode(
            [normalize_text(text) for text in texts],
            encoder_type="question_encoder",
        )
        similarity = np.inner(query_vectors, self.response_vectors)
        combined_results = []
        for row_idx, text in enumerate(texts):
            bm25_scores = self.bm25_model.score_vector(text)
            bm25_norm = bm25_scores / (bm25_scores.max() + 1e-9)
            sim_scores = similarity[row_idx]
            sim_norm = sim_scores / (sim_scores.max() + 1e-9)
            combined = self.bm25_weight * bm25_norm + (1 - self.bm25_weight) * sim_norm
            k = min(top_k, combined.shape[0])
            top_indices = np.argsort(-combined)[:k]
            combined_results.append(
                [self.bm25_model.subcategories[idx] for idx in top_indices]
            )
        return combined_results

In [ ]:
bm25_models = {
    label: BM25Retriever(subcats) for label, subcats in label_subcategories.items()
}
hybrid_models = {
    label: HybridRetriever(usem_models[label], bm25_models[label], bm25_weight=0.4)
    for label in usem_models
}

In [ ]:
baseline_variants = {
    "Random": random_models,
    "BM25": bm25_models,
    "USEM + BM25 hybrid": hybrid_models,
}

variant_records = {}
for variant, models in baseline_variants.items():
    records = retrieve_samples(samples_by_label, models, top_k=5)
    variant_records[variant] = records

variant_accuracy = {
    variant: compute_topk_accuracy(records)
    for variant, records in variant_records.items()
}
variant_pivots = {
    variant: df.pivot(index="label", columns="k", values="accuracy").sort_index()
    for variant, df in variant_accuracy.items()
}

for variant, pivot in variant_pivots.items():
    display(Markdown(f"**{variant} baseline top-k accuracy**"))
    display(pivot)

    fig, ax = plt.subplots()

    sns.heatmap(
        pivot.loc[
            [label for label in baseline_accuracy_pivot.index if label != "OVERALL"]
        ],
        annot=True,
        fmt=".2f",
        cmap="YlGnBu",
        ax=ax,
    )
    ax.set_title(f"{variant} baseline top-k accuracy by label")
    ax.set_xlabel("k")
    ax.set_ylabel("Label")
    plt.tight_layout()
    plt.show()

## Challenger models: PyTorch sentence-transformers

We now evaluate two PyTorch-based encoders as potential replacements for the TensorFlow USEM model:
- **DistilUSE**: Knowledge-distilled USE producing 512-dimensional embeddings
- **MultilingualMiniLM**: Higher-quality MiniLM producing 384-dimensional embeddings

Both models support Apple Silicon (M1/M2/M3) via MPS backend and eliminate the TensorFlow dependency.

In [ ]:
from aymurai.models.usem.sentence_transformers_encoder import (
    DistilUSEEncoder,
    MultilingualMiniLMEncoder,
)

# Initialize challenger encoders
distiluse_encoder = DistilUSEEncoder()
minilm_encoder = MultilingualMiniLMEncoder()

challenger_encoders = {
    "DistilUSE": distiluse_encoder,
    "MiniLM": minilm_encoder,
}

In [ ]:
from hashlib import md5

from aymurai.utils.download import download
from aymurai.utils.misc import is_url


class SentenceTransformerRetriever:
    def __init__(
        self,
        label,
        subcategories_path,
        encoder,
        encoder_name="challenger",
        batch_size=256,
    ):
        self.label = label
        self.encoder = encoder
        self.batch_size = batch_size
        self.encoder_name = encoder_name
        self.subcategories = self._load_subcategories(subcategories_path)
        normalized_subcategories = [
            normalize_subcategory(sub) for sub in self.subcategories
        ]
        self.response_vectors = self.encoder.batch_encode(
            normalized_subcategories,
            encoder_type="response_encoder",
            batch_size=batch_size,
        )

    def _load_subcategories(self, path):
        resolved_path = Path(path)
        if is_url(path):
            cache_root = Path(
                os.getenv("AYMURAI_CACHE_BASEPATH", "/resources/cache/aymurai")
            )
            target_dir = cache_root / "sentence_transformer_retriever"
            target_dir.mkdir(parents=True, exist_ok=True)
            target_path = target_dir / md5(path.encode("utf-8")).hexdigest()
            resolved_path = Path(download(path, str(target_path)))
        if not resolved_path.exists():
            raise FileNotFoundError(
                f"Subcategory file not found for {self.label}: {resolved_path}"
            )
        with resolved_path.open() as file:
            return [line.strip() for line in file if line.strip()]

    def batch_retrieve(self, texts, top_k=5):
        if not texts:
            return []

        query_vectors = self.encoder.batch_encode(
            texts,
            encoder_type="question_encoder",
            batch_size=self.batch_size,
        )
        similarity = np.inner(query_vectors, self.response_vectors)
        k = min(top_k, similarity.shape[1])
        top_indices = np.argsort(-similarity, axis=1)[:, :k]
        return [[self.subcategories[idx] for idx in row] for row in top_indices]


class HybridSentenceTransformerRetriever:
    def __init__(self, base_retriever, bm25_model, bm25_weight: float = 0.4):
        self.base_retriever = base_retriever
        self.bm25_model = bm25_model
        self.bm25_weight = bm25_weight
        self.encoder = base_retriever.encoder
        self.response_vectors = base_retriever.response_vectors
        self.subcategories = base_retriever.subcategories
        self.batch_size = base_retriever.batch_size

    def batch_retrieve(self, texts, top_k=5):
        if not texts:
            return []

        query_vectors = self.encoder.batch_encode(
            texts,
            encoder_type="question_encoder",
            batch_size=self.batch_size,
        )
        similarity = np.inner(query_vectors, self.response_vectors)
        combined_results = []
        for row_idx, text in enumerate(texts):
            bm25_scores = self.bm25_model.score_vector(text)
            bm25_norm = bm25_scores / (bm25_scores.max() + 1e-9)
            sim_scores = similarity[row_idx]
            sim_norm = sim_scores / (sim_scores.max() + 1e-9)
            combined = self.bm25_weight * bm25_norm + (1 - self.bm25_weight) * sim_norm
            k = min(top_k, combined.shape[0])
            top_indices = np.argsort(-combined)[:k]
            combined_results.append([self.subcategories[idx] for idx in top_indices])
        return combined_results


def build_challenger_models(usem_configs, encoder, encoder_name, batch_size=256):
    return {
        label: SentenceTransformerRetriever(
            label=label,
            subcategories_path=cfg["subcategories_path"],
            encoder=encoder,
            encoder_name=encoder_name,
            batch_size=batch_size,
        )
        for label, cfg in usem_configs.items()
    }


def build_hybrid_sentence_transformer_models(base_models, bm25_models, bm25_weight=0.4):
    return {
        label: HybridSentenceTransformerRetriever(
            base_retriever=base_models[label],
            bm25_model=bm25_models[label],
            bm25_weight=bm25_weight,
        )
        for label in base_models
    }


def evaluate_sentence_transformers(samples, models, top_k=5):
    records = []
    for label, items in samples.items():
        retriever = models[label]
        texts = [item["text"] for item in items]
        retrieved_lists = retriever.batch_retrieve(texts, top_k=top_k)
        for item, retrieved in zip(items, retrieved_lists):
            records.append(
                {
                    "label": label,
                    "text": item["text"],
                    "choices": item["choices"],
                    "retrieved": retrieved,
                }
            )
    return records


distiluse_models = build_challenger_models(usem_configs, distiluse_encoder, "DistilUSE")
minilm_models = build_challenger_models(usem_configs, minilm_encoder, "MiniLM")

### DistilUSE evaluation

In [ ]:
distiluse_retrieval_records = evaluate_sentence_transformers(
    samples_by_label,
    distiluse_models,
    top_k=5,
)
distiluse_accuracy = compute_topk_accuracy(distiluse_retrieval_records)
distiluse_accuracy_pivot = distiluse_accuracy.pivot(
    index="label", columns="k", values="accuracy"
).sort_index()

distiluse_accuracy_pivot

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    distiluse_accuracy_pivot.loc[
        [label for label in distiluse_accuracy_pivot.index if label != "OVERALL"]
    ],
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    ax=ax,
)
ax.set_title("DistilUSE top-k accuracy by label")
ax.set_xlabel("k")
ax.set_ylabel("Label")
plt.tight_layout()
plt.show()

distiluse_accuracy_pivot.loc[["OVERALL"]]

### MultilingualMiniLM evaluation

In [ ]:
minilm_retrieval_records = evaluate_sentence_transformers(
    samples_by_label,
    minilm_models,
    top_k=5,
)
minilm_accuracy = compute_topk_accuracy(minilm_retrieval_records)
minilm_accuracy_pivot = minilm_accuracy.pivot(
    index="label", columns="k", values="accuracy"
).sort_index()

minilm_accuracy_pivot

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    minilm_accuracy_pivot.loc[
        [label for label in minilm_accuracy_pivot.index if label != "OVERALL"]
    ],
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    ax=ax,
)
ax.set_title("MultilingualMiniLM top-k accuracy by label")
ax.set_xlabel("k")
ax.set_ylabel("Label")
plt.tight_layout()
plt.show()

minilm_accuracy_pivot.loc[["OVERALL"]]

### Hybrid (BM25 + encoder) evaluation

In [ ]:
distiluse_hybrid_models = build_hybrid_sentence_transformer_models(
    distiluse_models,
    bm25_models,
    bm25_weight=0.4,
)
minilm_hybrid_models = build_hybrid_sentence_transformer_models(
    minilm_models,
    bm25_models,
    bm25_weight=0.4,
)

distiluse_hybrid_records = evaluate_sentence_transformers(
    samples_by_label,
    distiluse_hybrid_models,
    top_k=5,
)
distiluse_hybrid_accuracy = compute_topk_accuracy(distiluse_hybrid_records)
distiluse_hybrid_pivot = distiluse_hybrid_accuracy.pivot(
    index="label", columns="k", values="accuracy"
).sort_index()

distiluse_hybrid_pivot

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    distiluse_hybrid_pivot.loc[
        [label for label in distiluse_hybrid_pivot.index if label != "OVERALL"]
    ],
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    ax=ax,
)
ax.set_title("DistilUSE + BM25 hybrid top-k accuracy by label")
ax.set_xlabel("k")
ax.set_ylabel("Label")
plt.tight_layout()
plt.show()

distiluse_hybrid_pivot.loc[["OVERALL"]]

In [ ]:
minilm_hybrid_records = evaluate_sentence_transformers(
    samples_by_label,
    minilm_hybrid_models,
    top_k=5,
)
minilm_hybrid_accuracy = compute_topk_accuracy(minilm_hybrid_records)
minilm_hybrid_pivot = minilm_hybrid_accuracy.pivot(
    index="label", columns="k", values="accuracy"
).sort_index()

minilm_hybrid_pivot

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    minilm_hybrid_pivot.loc[
        [label for label in minilm_hybrid_pivot.index if label != "OVERALL"]
    ],
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    ax=ax,
)
ax.set_title("MiniLM + BM25 hybrid top-k accuracy by label")
ax.set_xlabel("k")
ax.set_ylabel("Label")
plt.tight_layout()
plt.show()

minilm_hybrid_pivot.loc[["OVERALL"]]

## Model comparison and champion selection

We compare the three models across all metrics to select the best replacement for the TensorFlow USEM implementation.

In [ ]:
# Consolidate OVERALL accuracy across all models
comparison_df = pd.DataFrame(
    {
        "TensorFlow USEM": baseline_accuracy_pivot.loc["OVERALL"],
        "DistilUSE": distiluse_accuracy_pivot.loc["OVERALL"],
        "MiniLM": minilm_accuracy_pivot.loc["OVERALL"],
        "TensorFlow USEM + BM25": variant_pivots["USEM + BM25 hybrid"].loc["OVERALL"],
        "DistilUSE + BM25": distiluse_hybrid_pivot.loc["OVERALL"],
        "MiniLM + BM25": minilm_hybrid_pivot.loc["OVERALL"],
    }
).T

comparison_df

In [ ]:
# Visualize comparison across k values as bar chart, sorted by performance at each k
fig, ax = plt.subplots(figsize=(10, 6))
melted = comparison_df.reset_index().melt(
    id_vars="index", var_name="k", value_name="accuracy"
)

sns.barplot(
    data=melted.sort_values(by="accuracy", ascending=False),
    x="k",
    y="accuracy",
    hue="index",
    ax=ax,
)

ax.set_title("Top-k Accuracy Comparison Across Models")
ax.set_xlabel("k")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize comparison across k values as bar chart, sorted by performance at each k
fig, ax = plt.subplots(figsize=(10, 6))
melted = comparison_df.reset_index().melt(
    id_vars="index", var_name="k", value_name="accuracy"
)
# Drop TF USEM + BM25 variant for clarity
melted = melted[melted["index"] != "TensorFlow USEM + BM25"]

sns.barplot(
    data=melted.sort_values(by="accuracy", ascending=False),
    x="k",
    y="accuracy",
    hue="index",
    ax=ax,
)

ax.set_title("Top-k Accuracy Comparison Across Models")
ax.set_xlabel("k")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Per-label comparison for top-1 accuracy
label_index = [label for label in baseline_accuracy_pivot.index if label != "OVERALL"]
label_comparison = pd.DataFrame(
    {
        "TensorFlow USEM": baseline_accuracy_pivot.loc[label_index, 1],
        "DistilUSE": distiluse_accuracy_pivot.loc[label_index, 1],
        "MiniLM": minilm_accuracy_pivot.loc[label_index, 1],
        "TensorFlow USEM + BM25": variant_pivots["USEM + BM25 hybrid"].loc[
            label_index, 1
        ],
        "DistilUSE + BM25": distiluse_hybrid_pivot.loc[label_index, 1],
        "MiniLM + BM25": minilm_hybrid_pivot.loc[label_index, 1],
    }
)

label_comparison

In [ ]:
# Visualize comparison across labels as bar chart, sorted by performance at each label
fig, ax = plt.subplots(figsize=(10, 6))

melted = label_comparison.reset_index().melt(
    id_vars="label", var_name="Model", value_name="accuracy"
)

sns.barplot(
    data=melted.sort_values(by=["label", "accuracy"], ascending=False),
    x="label",
    y="accuracy",
    hue="Model",
    ax=ax,
)

ax.set_title("Top-1 Accuracy Comparison by Label")
ax.set_xlabel("Label")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.legend(title="Model", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize comparison across labels as bar chart, sorted by performance at each label
fig, ax = plt.subplots(figsize=(10, 6))

melted = label_comparison.reset_index().melt(
    id_vars="label", var_name="Model", value_name="accuracy"
)

for label in melted["label"].unique():
    subset = melted[melted["label"] == label].sort_values(
        by="accuracy", ascending=False
    )
    sns.barplot(
        data=subset,
        x="label",
        y="accuracy",
        hue="Model",
        ax=ax,
    )

# Deduplicate legend entries and draw once after plotting
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(
    by_label.values(),
    by_label.keys(),
    title="Model",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
)

ax.set_title("Top-1 Accuracy Comparison by Label")
ax.set_xlabel("Label")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize comparison across labels as bar chart, sorted by performance at each label
fig, ax = plt.subplots(figsize=(10, 6))

melted = label_comparison.reset_index().melt(
    id_vars="label", var_name="Model", value_name="accuracy"
)

# Drop TF USEM + BM25 variant for clarity
melted = melted[melted["Model"] != "TensorFlow USEM + BM25"]

for label in melted["label"].unique():
    subset = melted[melted["label"] == label].sort_values(
        by="accuracy", ascending=False
    )
    sns.barplot(
        data=subset,
        x="label",
        y="accuracy",
        hue="Model",
        ax=ax,
    )

# Deduplicate legend entries and draw once after plotting
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(
    by_label.values(),
    by_label.keys(),
    title="Model",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
)

ax.set_title("Top-1 Accuracy Comparison by Label")
ax.set_xlabel("Label")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Determine champion model
top1_scores = comparison_df[1]
champion_model = top1_scores.idxmax()
champion_score = top1_scores.max()

# Calculate performance delta vs baseline
baseline_score = top1_scores["TensorFlow USEM"]
challengers = top1_scores.drop("TensorFlow USEM")

performance_summary = pd.DataFrame(
    {
        "Model": comparison_df.index,
        "Top-1": comparison_df[1].values,
        "Top-3": comparison_df[3].values,
        "Top-5": comparison_df[5].values,
        "Δ vs Baseline (Top-1)": [
            0.0 if model == "TensorFlow USEM" else (score - baseline_score)
            for model, score in zip(comparison_df.index, comparison_df[1].values)
        ],
    }
).set_index("Model")

performance_summary

In [ ]:
final_summary_md = f"""
## Champion Model Selection

### Performance Summary

{performance_summary.to_markdown()}

### Winner: **{champion_model}**

- **Top-1 Accuracy**: {champion_score:.2%}
- **Performance vs Baseline**: {(champion_score - baseline_score):.2%} ({"+" if champion_score >= baseline_score else ""}{((champion_score - baseline_score) / baseline_score * 100):.1f}%)

### Key Findings

1. **{champion_model}** achieves the highest top-1 accuracy at {champion_score:.2%}
2. All PyTorch models eliminate TensorFlow dependency and support Apple Silicon (MPS)
3. Both challenger models maintain competitive accuracy while offering:
   - Cross-platform compatibility (Linux, macOS, Windows)
   - Apple Silicon acceleration via MPS backend
   - Smaller model footprint and faster inference

### Implementation Recommendations

1. **Replace TensorFlow USEM** with **{champion_model}** in production pipeline
2. Update `USEMSubcategorizer` to use the selected PyTorch encoder by default
3. Regenerate cached embeddings using the champion encoder
4. Update deployment documentation to reflect new dependencies
5. Verify inference latency and memory footprint in production environment

### Next Steps

- [ ] Implement {champion_model} as default encoder in `aymurai.models.usem`
- [ ] Regenerate response embeddings for all label categories
- [ ] Update pipeline configuration files
- [ ] Run integration tests across all supported platforms
- [ ] Update deployment scripts and documentation
"""

Markdown(final_summary_md)

In [ ]:
# Best-performing model per label and k (excluding TensorFlow USEM + BM25)
label_index = [label for label in baseline_accuracy_pivot.index if label != "OVERALL"]
ks = [k for k in range(1, 6) if k in baseline_accuracy_pivot.columns]

model_pivots = {
    "TensorFlow USEM": baseline_accuracy_pivot,
    "DistilUSE": distiluse_accuracy_pivot,
    "MiniLM": minilm_accuracy_pivot,
    "DistilUSE + BM25": distiluse_hybrid_pivot,
    "MiniLM + BM25": minilm_hybrid_pivot,
}

records = []
for model_name, pivot in model_pivots.items():
    for label in label_index:
        for k in ks:
            records.append(
                {
                    "label": label,
                    "k": k,
                    "model": model_name,
                    "accuracy": pivot.loc[label, k],
                }
            )

scores = pd.DataFrame(records)
best_models = scores.loc[scores.groupby(["label", "k"])["accuracy"].idxmax()]
model_ranking = (
    best_models["model"].value_counts().rename_axis("model").reset_index(name="wins")
)

best_models.reset_index(drop=True)
model_ranking

In [ ]:
scores.query("k == 1").sort_values(["label", "accuracy"], ascending=[True, False])

In [ ]:
scores.query("k == 3").sort_values(["label", "accuracy"], ascending=[True, False])

In [ ]:
scores.query("k == 5").sort_values(["label", "accuracy"], ascending=[True, False])

In [ ]:
# Grid search BM25 weights for DistilUSE and MiniLM hybrids
weight_grid = [round(w, 2) for w in np.linspace(0.0, 0.9, 10)]
weight_grid

In [ ]:
search_results = []
pivot_lookup = {}

# Cache texts per label
label_texts = {
    label: [item["text"] for item in items] for label, items in samples_by_label.items()
}

# Cache BM25 score vectors per label to avoid recomputing across weights
bm25_cache = {
    label: [bm25_models[label].score_vector(text) for text in texts]
    for label, texts in label_texts.items()
}

# Cache question embeddings per model and label
query_cache = {}
for model_name, base_models in {
    "DistilUSE": distiluse_models,
    "MiniLM": minilm_models,
}.items():
    any_model = next(iter(base_models.values()))
    encoder = any_model.encoder
    batch_size = any_model.batch_size
    query_cache[model_name] = {
        label: encoder.batch_encode(
            texts, encoder_type="question_encoder", batch_size=batch_size
        )
        for label, texts in label_texts.items()
    }


def hybrid_retrieve_cached(
    base_retriever,
    bm25_model,
    texts,
    query_vectors,
    bm25_scores,
    bm25_weight,
    top_k=5,
):
    similarity = np.inner(query_vectors, base_retriever.response_vectors)
    combined_results = []
    for idx, text in enumerate(texts):
        bm25_vec = bm25_scores[idx]
        bm25_norm = bm25_vec / (bm25_vec.max() + 1e-9)
        sim_scores = similarity[idx]
        sim_norm = sim_scores / (sim_scores.max() + 1e-9)
        combined = bm25_weight * bm25_norm + (1 - bm25_weight) * sim_norm
        k = min(top_k, combined.shape[0])
        top_indices = np.argsort(-combined)[:k]
        combined_results.append([bm25_model.subcategories[i] for i in top_indices])
    return combined_results


for model_name, base_models in {
    "DistilUSE": distiluse_models,
    "MiniLM": minilm_models,
}.items():
    for weight in weight_grid:
        records = []
        for label, base_retriever in base_models.items():
            texts = label_texts[label]
            query_vectors = query_cache[model_name][label]
            bm25_scores = bm25_cache[label]
            retrieved_lists = hybrid_retrieve_cached(
                base_retriever=base_retriever,
                bm25_model=bm25_models[label],
                texts=texts,
                query_vectors=query_vectors,
                bm25_scores=bm25_scores,
                bm25_weight=weight,
                top_k=5,
            )
            for item, retrieved in zip(samples_by_label[label], retrieved_lists):
                records.append(
                    {
                        "label": label,
                        "text": item["text"],
                        "choices": item["choices"],
                        "retrieved": retrieved,
                    }
                )
        accuracy = compute_topk_accuracy(records)
        pivot = accuracy.pivot(
            index="label", columns="k", values="accuracy"
        ).sort_index()
        pivot_lookup[(model_name, weight)] = pivot
        overall = pivot.loc["OVERALL"]
        search_results.append(
            {
                "model": model_name,
                "bm25_weight": weight,
                "top1": overall.get(1, np.nan),
                "top3": overall.get(3, np.nan),
                "top5": overall.get(5, np.nan),
            }
        )

search_df = pd.DataFrame(search_results)

# Best weight per model by top-1 accuracy
best_rows = search_df.loc[search_df.groupby("model")["top1"].idxmax()].reset_index(
    drop=True
)
best_pivots = {
    row.model: pivot_lookup[(row.model, row.bm25_weight)]
    for _, row in best_rows.iterrows()
}

best_rows

In [ ]:
search_df

### Hybrid (tuned BM25 + encoder) evaluation

In [ ]:
# Evaluate hybrids at tuned BM25 weights

tuned_records = {}
tuned_pivots = {}

for row in best_rows.itertuples():
    model_name = row.model
    weight = row.bm25_weight
    base_models = distiluse_models if model_name == "DistilUSE" else minilm_models

    records = []
    for label, base_retriever in base_models.items():
        texts = label_texts[label]
        query_vectors = query_cache[model_name][label]
        bm25_scores = bm25_cache[label]
        retrieved_lists = hybrid_retrieve_cached(
            base_retriever=base_retriever,
            bm25_model=bm25_models[label],
            texts=texts,
            query_vectors=query_vectors,
            bm25_scores=bm25_scores,
            bm25_weight=weight,
            top_k=5,
        )
        for item, retrieved in zip(samples_by_label[label], retrieved_lists):
            records.append(
                {
                    "label": label,
                    "text": item["text"],
                    "choices": item["choices"],
                    "retrieved": retrieved,
                }
            )

    accuracy = compute_topk_accuracy(records)
    pivot = accuracy.pivot(index="label", columns="k", values="accuracy").sort_index()
    tuned_records[model_name] = records
    tuned_pivots[model_name] = pivot

# Show tuned weights and overall performance
best_rows_display = best_rows.copy()
for model_name, pivot in tuned_pivots.items():
    best_rows_display.loc[best_rows_display["model"] == model_name, "top1"] = pivot.loc[
        "OVERALL", 1
    ]
    best_rows_display.loc[best_rows_display["model"] == model_name, "top3"] = pivot.loc[
        "OVERALL", 3
    ]
    best_rows_display.loc[best_rows_display["model"] == model_name, "top5"] = pivot.loc[
        "OVERALL", 5
    ]

best_rows_display

In [ ]:
# Full pipeline re-evaluation with evaluate_sentence_transformers at tuned weights

best_weights = best_rows.set_index("model")["bm25_weight"].to_dict()

# Build tuned hybrid retrievers (standard pipeline, no caching shortcuts)
tuned_hybrid_models = {
    "DistilUSE + BM25 tuned": build_hybrid_sentence_transformer_models(
        distiluse_models,
        bm25_models,
        bm25_weight=float(best_weights["DistilUSE"]),
    ),
    "MiniLM + BM25 tuned": build_hybrid_sentence_transformer_models(
        minilm_models, bm25_models, bm25_weight=float(best_weights["MiniLM"])
    ),
}

# Run evaluation pipeline
attribution = {}
tuned_hybrid_records = {}
tuned_hybrid_accuracy = {}
tuned_hybrid_pivots = {}

for name, models in tuned_hybrid_models.items():
    records = evaluate_sentence_transformers(samples_by_label, models, top_k=5)
    tuned_hybrid_records[name] = records
    df = compute_topk_accuracy(records)
    tuned_hybrid_accuracy[name] = df
    tuned_hybrid_pivots[name] = df.pivot(
        index="label", columns="k", values="accuracy"
    ).sort_index()

# Overall comparison table including tuned hybrids
comparison_df_tuned = pd.DataFrame(
    {
        "TensorFlow USEM": baseline_accuracy_pivot.loc["OVERALL"],
        "TensorFlow USEM + BM25": variant_pivots["USEM + BM25 hybrid"].loc["OVERALL"],
        "DistilUSE": distiluse_accuracy_pivot.loc["OVERALL"],
        "MiniLM": minilm_accuracy_pivot.loc["OVERALL"],
        "DistilUSE + BM25 tuned": tuned_hybrid_pivots["DistilUSE + BM25 tuned"].loc[
            "OVERALL"
        ],
        "MiniLM + BM25 tuned": tuned_hybrid_pivots["MiniLM + BM25 tuned"].loc[
            "OVERALL"
        ],
    }
).T

comparison_df_tuned.sort_values(by=1, ascending=False)

In [ ]:
# Heatmaps per label for each model
pivot_map = {
    "TensorFlow USEM": baseline_accuracy_pivot,
    "TensorFlow USEM + BM25": variant_pivots["USEM + BM25 hybrid"],
    "DistilUSE": distiluse_accuracy_pivot,
    "MiniLM": minilm_accuracy_pivot,
    "DistilUSE + BM25 tuned": tuned_hybrid_pivots["DistilUSE + BM25 tuned"],
    "MiniLM + BM25 tuned": tuned_hybrid_pivots["MiniLM + BM25 tuned"],
}

labels_no_overall = [label for label in label_index]

for name, pivot in pivot_map.items():
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(
        pivot.loc[labels_no_overall],
        annot=True,
        fmt=".2f",
        cmap="YlGnBu",
        vmin=0,
        vmax=1,
        ax=ax,
    )
    ax.set_title(f"{name} top-k accuracy by label")
    ax.set_xlabel("k")
    ax.set_ylabel("Label")
    plt.tight_layout()
    plt.show()